# Composite Design Pattern 

explained using the classic File System (Files and Folders) example.

#### The Concept
The Composite Pattern allows you to compose objects into tree structures to represent part-whole hierarchies. Crucial Point: It lets clients treat individual objects (Files) and compositions of objects (Folders) uniformly.

The client code shouldn't care if it's "opening" a single file or a folder containing 100 files; the command is the same.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use an Abstract Base Class (ABC) to define the common interface. Both the Leaf (File) and the Composite (Folder) must inherit from this.

#### THE COMPONENT (The Common Interface)

In [1]:
from abc import ABC, abstractmethod

class FileSystemComponent(ABC):
    """
    Defines the interface for objects in the composition.
    Both Files and Folders must implement 'show_details'.
    """
    @abstractmethod
    def show_details(self) -> None:
        pass

#### THE LEAF (The File)

In [2]:
class File(FileSystemComponent):
    def __init__(self, name: str):
        self.name = name

    def show_details(self) -> None:
        print(f"    - File: {self.name}")

#### THE COMPOSITE (The Folder)

In [5]:
from typing import List

class Folder(FileSystemComponent):
    def __init__(self, name: str):
        self.name = name
        # The list holds children, which can be Files OR other Folders
        self.children: List[FileSystemComponent] = []

    def add(self, component: FileSystemComponent):
        self.children.append(component)

    def remove(self, component: FileSystemComponent):
        self.children.remove(component)

    def show_details(self) -> None:
        print(f"+ Folder: {self.name}")
        # RECURSION: Delegate the task to children
        for child in self.children:
            child.show_details()

#### CLIENT CODE

In [6]:
def main():
    # 1. Create Leafs
    file1 = File("resume.pdf")
    file2 = File("photo.png")
    file3 = File("todo.txt")

    # 2. Create Composites
    folder_docs = Folder("Documents")
    folder_music = Folder("Music")
    root_folder = Folder("C: Drive")

    # 3. Compose the Tree
    folder_docs.add(file1)
    folder_docs.add(file2)
    
    # Nested Composite (Folder inside Folder)
    root_folder.add(folder_docs)
    root_folder.add(folder_music)
    root_folder.add(file3) # Adding file directly to root

    # 4. Treat them uniformly
    print("--- Java-Style Recursive Tree ---")
    root_folder.show_details()

if __name__ == "__main__":
    main()

--- Java-Style Recursive Tree ---
+ Folder: C: Drive
+ Folder: Documents
    - File: resume.pdf
    - File: photo.png
+ Folder: Music
    - File: todo.txt


## The Pythonic Way

In Python, we can make this cleaner using **Dataclasses** and **Duck Typing (or Protocols)**. We can also implement standard magic methods like `__iter__` or `__str__` to make the objects behave like native Python types.

We drop the rigid Abstract Base Class inheritance if we trust the interfaces match (Duck Typing), but using Protocol is safer practice in modern Python.

#### PROTOCOL (Implicit Interface)

In [8]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Component(Protocol):
    def render(self, indent: int = 0) -> str: ...

#### THE LEAF (File)

In [14]:
from dataclasses import dataclass

@dataclass
class File:
    name: str
    size_kb: int

    def render(self, indent: int = 0) -> str:
        # Returns a string instead of printing (more functional)
        return f"{' ' * indent}📄 {self.name} ({self.size_kb}kb)"

#### THE COMPOSITE (Folder)

In [16]:
from dataclasses import dataclass, field

@dataclass
class Folder:
    name: str
    # Automatically initialize an empty list
    children: List[Component] = field(default_factory=list)

    def add(self, *items: Component):
        """Allows adding multiple items at once: add(f1, f2, f3)"""
        self.children.extend(items)

    def render(self, indent: int = 0) -> str:
        # 1. Render Self
        lines = [f"{' ' * indent}📁 {self.name}/"]
        
        # 2. Recursive Step (List Comprehension)
        # We increase indent for children
        for child in self.children:
            lines.append(child.render(indent + 4))
            
        return "\n".join(lines)
    
    # PYTHON MAGIC: Make the folder iterable!
    # This allows: 'for item in my_folder:'
    def __iter__(self):
        return iter(self.children)

#### CLIENT CODE

In [17]:
def main():
    # 1. Setup Tree
    root = Folder("Project_X")
    src = Folder("Source")
    
    # Pythonic Variadic Add
    src.add(
        File("main.py", 12),
        File("utils.py", 8)
    )

    root.add(
        src,
        File("README.md", 2),
        File("requirements.txt", 1)
    )

    # 2. Usage
    print("--- Pythonic Tree Render ---")
    print(root.render())

    # 3. Iteration Magic (Thanks to __iter__)
    print("\n--- Iterating Root Direct Children ---")
    for item in root:
        print(f"Found: {item.name}")

if __name__ == "__main__":
    main()

--- Pythonic Tree Render ---
📁 Project_X/
    📁 Source/
        📄 main.py (12kb)
        📄 utils.py (8kb)
    📄 README.md (2kb)
    📄 requirements.txt (1kb)

--- Iterating Root Direct Children ---
Found: Source
Found: README.md
Found: requirements.txt


#### Key Pythonic Features Used
- `@dataclass`: Removes the boilerplate `__init__` code.
- `*items (Args)`: The add method accepts variable arguments, making tree construction look cleaner (`src.add(f1, f2)`).
- `__iter__`: By implementing this, the Folder becomes iterable. You can loop over it just like a list.
- String Building: Instead of methods that `print (side effect)`, we return strings, which is more testable and functional.

#### Summary

- Java-like: Focuses on strict type safety and hierarchy (`extends Component`). Useful for large teams enforcing rigid structures.
- Pythonic: Focuses on usability and readability (`dataclasses, iterables`). It treats the structure as data.